# 02 · Modelización predictiva sin fuga de información

Este notebook entrena modelos para predecir `risk_class_1h`: 0 = estable, 1 = riesgo de vaciado y 2 = riesgo de saturación. Solo utiliza filas con una etiqueta disponible y los conjuntos temporales `train`, `validation` o `test`.

La regla clave es que el imputador numérico y el codificador de categorías se ajustan exclusivamente con `train`. Validation y test solo se transforman con esos objetos ya aprendidos.

## Preparación del entorno

El notebook requiere `pandas`, `numpy` y `scikit-learn`. Si tu kernel no tiene scikit-learn, descomenta la siguiente línea y ejecútala una sola vez.

In [ ]:
# Descomenta esta línea únicamente si scikit-learn no está instalado en tu entorno Jupyter.
#pip install scikit-learn

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight

# Encontramos la raíz del proyecto tanto si Jupyter se abre en la raíz como en notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Datos modelado').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_DIR = PROJECT_ROOT / 'Datos modelado' / 'estacion_hora_features'
TARGET = 'risk_class_1h'

# Para una primera ejecución reproducible en un ordenador personal se usa una muestra estratificada.
# Cambia este valor a None si dispones de recursos para entrenar con todas las filas de train.
MAX_TRAIN_ROWS = 750_000
RANDOM_STATE = 42

# El test se mantiene bloqueado hasta decidir el modelo mirando solo validation.
EVALUATE_FINAL_TEST = True


## Selección de variables

No se incluyen `future_occupancy_ratio_1h`, `risk_empty_1h`, `risk_full_1h` ni la propia etiqueta como predictores porque pertenecen al futuro o definen el objetivo. Las variables de flujo disponibles ya vienen retrasadas desde el notebook anterior.

In [ ]:
# Variables numéricas conocidas en el instante de predicción.
NUMERIC_FEATURES = [
    'capacity', 'bikes_available', 'docks_available', 'reservations_count',
    'occupancy_ratio', 'light', 'weather_available',
    'uv_radiation_median_mw_m2', 'wind_speed_median_m_s',
    'wind_direction_sin_mean', 'wind_direction_cos_mean',
    'temperature_median_c', 'relative_humidity_median_pct',
    'barometric_pressure_median_mb', 'solar_radiation_median_w_m2',
    'precipitation_mean_l_m2', 'precipitation_max_l_m2',
    'n_temperature', 'n_relative_humidity', 'n_precipitation',
    'hour', 'day_of_week', 'month', 'week_of_year',
    'occupancy_ratio_lag_1h', 'occupancy_ratio_lag_2h', 'occupancy_ratio_lag_24h',
    'bikes_available_lag_1h', 'bikes_available_lag_2h', 'bikes_available_lag_24h',
    'net_flow_lag_1h', 'net_flow_lag_2h', 'net_flow_lag_24h',
    'departures_count_lag_1h', 'departures_count_lag_2h', 'departures_count_lag_24h',
    'arrivals_count_lag_1h', 'arrivals_count_lag_2h', 'arrivals_count_lag_24h',
    'occupancy_ratio_mean_previous_3h', 'occupancy_ratio_mean_previous_24h',
    'net_flow_mean_previous_3h', 'net_flow_mean_previous_24h',
    'departures_count_mean_previous_3h', 'departures_count_mean_previous_24h',
    'arrivals_count_mean_previous_3h', 'arrivals_count_mean_previous_24h',
]

# Las categorías describen la estación y el tipo de día sin imponer un orden artificial.
CATEGORICAL_FEATURES = ['station_id', 'tipo_dia']

# Leemos solo estas columnas para reducir memoria y evitamos usar cualquier columna futura.
COLUMNS_TO_LOAD = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET, 'dataset_split']
feature_files = sorted(FEATURE_DIR.glob('estacion_hora_features_*.csv'))
assert len(feature_files) == 48, f'Se esperaban 48 particiones y hay {len(feature_files)}'
print(f'Variables numéricas: {len(NUMERIC_FEATURES)}')
print(f'Variables categóricas: {len(CATEGORICAL_FEATURES)}')

In [ ]:
def count_train_rows() -> int:
    """Primera pasada ligera: cuenta solo las filas elegibles de train."""
    total = 0
    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=[TARGET, 'dataset_split'], chunksize=300_000):
            total += int((chunk['dataset_split'].eq('train') & chunk[TARGET].notna()).sum())
    return total


def load_modelling_data(train_fraction: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Carga solo filas elegibles y muestrea train dentro de cada bloque para ahorrar memoria."""
    train_parts, validation_parts, test_parts = [], [], []
    for file_path in feature_files:
        # chunksize evita cargar cada CSV mensual completo en memoria de una sola vez.
        for chunk in pd.read_csv(file_path, usecols=COLUMNS_TO_LOAD, chunksize=200_000, low_memory=False):
            keep = chunk['dataset_split'].isin(['train', 'validation', 'test']) & chunk[TARGET].notna()
            chunk = chunk.loc[keep].copy()
            if chunk.empty:
                continue
            train_chunk = chunk.loc[chunk['dataset_split'].eq('train')]
            if not train_chunk.empty:
                # El muestreo se hace dentro de cada clase para conservar aproximadamente su proporción.
                if train_fraction < 1:
                    train_chunk = train_chunk.groupby(TARGET, group_keys=False).sample(frac=train_fraction, random_state=RANDOM_STATE)
                train_parts.append(train_chunk)
            validation_parts.append(chunk.loc[chunk['dataset_split'].eq('validation')])
            test_parts.append(chunk.loc[chunk['dataset_split'].eq('test')])
    return (
        pd.concat(train_parts, ignore_index=True),
        pd.concat(validation_parts, ignore_index=True),
        pd.concat(test_parts, ignore_index=True),
    )


train_total = count_train_rows()
train_fraction = 1.0 if MAX_TRAIN_ROWS is None else min(1.0, MAX_TRAIN_ROWS / train_total)
train, validation, test = load_modelling_data(train_fraction)

for frame in [train, validation, test]:
    frame[TARGET] = frame[TARGET].astype('int8')

print('Filas etiquetadas por conjunto:')
print({'train': len(train), 'validation': len(validation), 'test': len(test)})
print('Distribución de clases en train:')
print(train[TARGET].value_counts(normalize=True).sort_index())

In [ ]:
# Separamos X e y antes de ajustar cualquier transformador.
X_train = train[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train = train[TARGET]
X_validation = validation[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_validation = validation[TARGET]
X_test = test[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_test = test[TARGET]

# El imputador aprende las medianas exclusivamente con train.
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
])

# El codificador aprende las categorías existentes exclusivamente con train.
# handle_unknown='ignore' evita errores si validation o test contienen una categoría nueva.
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, NUMERIC_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
])

# IMPORTANTE: fit_transform se aplica solo sobre train; los otros conjuntos usan transform.
X_train_ready = preprocessor.fit_transform(X_train)
X_validation_ready = preprocessor.transform(X_validation)
X_test_ready = preprocessor.transform(X_test)

print(f'Columnas tras transformar: {X_train_ready.shape[1]}')

In [ ]:
def evaluate(model, features, target, name: str) -> dict:
    """Calcula métricas equilibradas para una clasificación multiclase desbalanceada."""
    prediction = model.predict(features)
    print(f'\n--- {name} ---')
    print('Balanced accuracy:', round(balanced_accuracy_score(target, prediction), 4))
    print('F1 macro:', round(f1_score(target, prediction, average='macro'), 4))
    print('Matriz de confusión (filas: real; columnas: predicción):')
    print(confusion_matrix(target, prediction, labels=[0, 1, 2]))
    print(classification_report(target, prediction, labels=[0, 1, 2], target_names=['estable', 'vaciado', 'saturación'], zero_division=0))
    return {
        'model': name,
        'balanced_accuracy': balanced_accuracy_score(target, prediction),
        'f1_macro': f1_score(target, prediction, average='macro'),
    }

# La línea base predice según la frecuencia de las clases y sirve para saber si el modelo aprende algo real.
dummy = DummyClassifier(strategy='prior', random_state=RANDOM_STATE)
dummy.fit(np.zeros((len(y_train), 1)), y_train)
baseline_validation = evaluate(dummy, np.zeros((len(y_validation), 1)), y_validation, 'Baseline: frecuencia de clases')

# La regresión logística es un primer modelo interpretable.
# Los pesos balanceados compensan que estable sea mucho más frecuente que las clases de riesgo.
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
logistic_model = LogisticRegression(
    solver='saga',
    max_iter=300,
    multi_class='multinomial',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
logistic_model.fit(X_train_ready, y_train, sample_weight=sample_weight)
logistic_validation = evaluate(logistic_model, X_validation_ready, y_validation, 'Regresión logística')

results_validation = pd.DataFrame([baseline_validation, logistic_validation]).sort_values('f1_macro', ascending=False)
results_validation

In [ ]:
# Ejecuta esta celda solo después de elegir el modelo usando validation.
# El test permanece fuera de todas las decisiones de selección y ajuste.
# Localizamos las carpetas del proyecto.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Datos modelado').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_DIR = PROJECT_ROOT / 'Datos modelado' / 'estacion_hora_features'
RESULTS_DIR = PROJECT_ROOT / 'Datos modelado' / 'resultados_modelos'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if EVALUATE_FINAL_TEST:
    test_results = evaluate(logistic_model, X_test_ready, y_test, 'Test final: regresión logística')
    pd.DataFrame([test_results]).to_csv(RESULTS_DIR / 'modelizacion_02_test_final.csv', index=False, encoding='utf-8-sig')
else:
    print('Test bloqueado. Cambia EVALUATE_FINAL_TEST a True después de elegir el modelo con validation.')

## Próximo paso

Tras establecer esta línea base, el siguiente notebook comparará modelos no lineales —por ejemplo, Random Forest o LightGBM si está disponible— usando exactamente las mismas divisiones temporales y el mismo protocolo sin fuga de información.